In [1]:
import json, h5py, numpy as np
from collections import Counter

DATA = "data"

with open(f"{DATA}/vg150/VG-SGG-dicts.json") as f:
    d = json.load(f)
idx_pred = {int(k): v for k, v in d['idx_to_predicate'].items()}

with h5py.File(f"{DATA}/vg150/VG-SGG.h5", "r") as f:
    preds = f['predicates'][:].flatten()
    first_r = f['img_to_first_rel'][:]
    last_r = f['img_to_last_rel'][:]

cnt = Counter(preds)
max_count = cnt.most_common(1)[0][1]
print(f"{len(preds)} predicates, {len(cnt)} unique")


622705 predicates, 50 unique


In [2]:
# top 50 predicates
for pid, count in cnt.most_common(50):
    name = idx_pred.get(pid, f"?{pid}")
    bar = "#" * int(count / max_count * 40)
    print(f"{name:20s} {bar} {count}")


on                   ######################################## 196495
has                  ################### 97473
wearing              ############## 71121
of                   ########## 53761
in                   ####### 37914
near                 ###### 30331
with                 ### 18428
behind               ### 18159
holding              ### 16514
above                ## 11602
sitting on           # 8003
wears                # 7471
under                # 6687
riding               # 6335
in front of          # 5609
standing on           4068
at                    3030
attached to           2689
carrying              2504
walking on            2343
over                  1871
belonging to          1518
for                   1507
looking at            1410
watching              1327
hanging from          1296
parked on             1162
laying on             1138
eating                1063
and                   933
covering              828
using                 822
between         

In [3]:
# distribution stats
counts = [c for _, c in cnt.most_common()]
for pct in [1, 5, 10, 25, 50, 75, 90, 100]:
    idx = int(len(counts) * pct / 100) - 1
    if idx >= 0:
        pid = cnt.most_common()[idx][0]
        name = idx_pred.get(pid, '?')
        print(f"p{pct:3d}: {counts[idx]:6d} ({name})")
print(f"\n>10000: {sum(1 for c in counts if c > 10000)}")
print(f">1000: {sum(1 for c in counts if c > 1000)}")
print(f"<100: {sum(1 for c in counts if c < 100)}")


p  5:  97473 (has)
p 10:  37914 (in)
p 25:   7471 (wears)
p 50:   1327 (watching)
p 75:    561 (lying on)
p 90:    296 (growing on)
p100:     30 (flying in)

>10000: 10
>1000: 29
<100: 2


In [4]:
# relationships per image
img_counts = last_r - first_r + 1
valid = img_counts[img_counts > 0]
print(f"rels per image: min={valid.min()} max={valid.max()} mean={valid.mean():.1f} median={np.median(valid):.0f}")
print(f"1 rel: {(valid==1).sum()}, 10+: {(valid>=10).sum()}, 50+: {(valid>=50).sum()}")


rels per image: min=1 max=272 mean=5.9 median=4
1 rel: 29793, 10+: 22307, 50+: 57
